In [ ]:

# OrchNAS: Orchestrated Neural Architecture Search Service

import copy
import math
import random
from dataclasses import dataclass
from typing import Dict, List, Tuple, Any

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset


def set_seed(seed: int = 42):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed(42)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")



# Search Space


SEARCH_SPACE = {
    "depth": [2, 3, 5],
    "width": [32, 64, 128, 192, 256],
    "activation": ["relu", "gelu", "tanh"],
    "op_type": ["standard", "depthwise", "inverted_bottleneck"],
    "kernel_size": [3, 5, 7],
}


def random_architecture() -> Dict[str, Any]:
    return {
        "depth": random.choice(SEARCH_SPACE["depth"]),
        "width": random.choice(SEARCH_SPACE["width"]),
        "activation": random.choice(SEARCH_SPACE["activation"]),
        "op_type": random.choice(SEARCH_SPACE["op_type"]),
        "kernel_size": random.choice(SEARCH_SPACE["kernel_size"]),
    }


def mutate_architecture(arch: Dict[str, Any], mutation_prob: float = 0.5) -> Dict[str, Any]:
    """Evolutionary mutation for candidate pool generation."""
    new_arch = copy.deepcopy(arch)
    for key, choices in SEARCH_SPACE.items():
        if random.random() < mutation_prob:
            current = new_arch[key]
            candidates = [c for c in choices if c != current]
            new_arch[key] = random.choice(candidates)
    return new_arch


def architecture_to_key(arch: Dict[str, Any]) -> Tuple:
    return (
        arch["depth"],
        arch["width"],
        arch["activation"],
        arch["op_type"],
        arch["kernel_size"],
    )

# Model building blocks

def get_activation(name: str):
    if name == "relu":
        return nn.ReLU(inplace=True)
    if name == "gelu":
        return nn.GELU()
    if name == "tanh":
        return nn.Tanh()
    raise ValueError(f"Unknown activation: {name}")


class StandardConvBlock(nn.Module):
    def __init__(self, in_ch: int, out_ch: int, kernel_size: int, activation: str):
        super().__init__()
        padding = kernel_size // 2
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=kernel_size, padding=padding, bias=False),
            nn.BatchNorm2d(out_ch),
            get_activation(activation),
        )

    def forward(self, x):
        return self.block(x)


class DepthwiseSeparableBlock(nn.Module):
    def __init__(self, in_ch: int, out_ch: int, kernel_size: int, activation: str):
        super().__init__()
        padding = kernel_size // 2
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, in_ch, kernel_size=kernel_size, padding=padding, groups=in_ch, bias=False),
            nn.BatchNorm2d(in_ch),
            get_activation(activation),
            nn.Conv2d(in_ch, out_ch, kernel_size=1, bias=False),
            nn.BatchNorm2d(out_ch),
            get_activation(activation),
        )

    def forward(self, x):
        return self.block(x)


class InvertedBottleneckBlock(nn.Module):
    def __init__(self, in_ch: int, out_ch: int, kernel_size: int, activation: str, expand_ratio: int = 2):
        super().__init__()
        hidden = max(in_ch * expand_ratio, out_ch)
        padding = kernel_size // 2
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, hidden, kernel_size=1, bias=False),
            nn.BatchNorm2d(hidden),
            get_activation(activation),
            nn.Conv2d(hidden, hidden, kernel_size=kernel_size, padding=padding, groups=hidden, bias=False),
            nn.BatchNorm2d(hidden),
            get_activation(activation),
            nn.Conv2d(hidden, out_ch, kernel_size=1, bias=False),
            nn.BatchNorm2d(out_ch),
        )

    def forward(self, x):
        return self.block(x)


def make_block(op_type: str, in_ch: int, out_ch: int, kernel_size: int, activation: str) -> nn.Module:
    if op_type == "standard":
        return StandardConvBlock(in_ch, out_ch, kernel_size, activation)
    if op_type == "depthwise":
        return DepthwiseSeparableBlock(in_ch, out_ch, kernel_size, activation)
    if op_type == "inverted_bottleneck":
        return InvertedBottleneckBlock(in_ch, out_ch, kernel_size, activation)
    raise ValueError(f"Unknown op_type: {op_type}")


class OrchNASModel(nn.Module):

    def __init__(self, arch: Dict[str, Any], num_classes: int = 10, in_channels: int = 3):
        super().__init__()
        self.arch = copy.deepcopy(arch)
        depth = arch["depth"]
        width = arch["width"]
        activation = arch["activation"]
        op_type = arch["op_type"]
        k = arch["kernel_size"]

        layers = []
        in_ch = in_channels
        for i in range(depth):
            out_ch = width
            layers.append(make_block(op_type, in_ch, out_ch, k, activation))
            if i < depth - 1:
                layers.append(nn.MaxPool2d(2))
            in_ch = out_ch

        self.backbone = nn.Sequential(*layers)
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.personal_head = nn.Linear(width, num_classes)

    def forward(self, x):
        x = self.backbone(x)
        x = self.pool(x)
        x = torch.flatten(x, 1)
        return self.personal_head(x)

    def forward_features(self, x):
        x = self.backbone(x)
        x = self.pool(x)
        x = torch.flatten(x, 1)
        return x


# Cost estimation: Params / FLOPs / Energy
def count_parameters(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def estimate_flops(arch: Dict[str, Any], image_size: int = 32, in_channels: int = 3, num_classes: int = 10) -> float:
    """
    Rough analytical FLOPs estimate.
    Enough for research simulation and energy-aware ranking.
    """
    depth = arch["depth"]
    width = arch["width"]
    op_type = arch["op_type"]
    k = arch["kernel_size"]

    h = image_size
    w = image_size
    in_ch = in_channels
    total_flops = 0.0

    for i in range(depth):
        out_ch = width

        if op_type == "standard":
            # Conv FLOPs ~ H * W * Cin * Cout * K * K
            total_flops += h * w * in_ch * out_ch * k * k
        elif op_type == "depthwise":
            total_flops += h * w * in_ch * k * k           # depthwise
            total_flops += h * w * in_ch * out_ch          # pointwise
        elif op_type == "inverted_bottleneck":
            hidden = max(in_ch * 2, out_ch)
            total_flops += h * w * in_ch * hidden          # expand 1x1
            total_flops += h * w * hidden * k * k          # depthwise
            total_flops += h * w * hidden * out_ch         # project 1x1

        if i < depth - 1:
            h = max(h // 2, 1)
            w = max(w // 2, 1)
        in_ch = out_ch

    total_flops += width * num_classes  # classifier
    return float(total_flops)


def estimate_energy(arch: Dict[str, Any], alpha: float, local_steps: int, image_size: int = 32) -> float:
    flops = estimate_flops(arch, image_size=image_size)
    return alpha * flops * local_steps


#dataset

def make_synthetic_client_dataset(
    n_samples: int = 256,
    num_classes: int = 10,
    image_size: int = 32,
    in_channels: int = 3,
    class_bias: int = 0,
):
    """
    Creates a simple synthetic non-IID dataset.
    """
    x = torch.randn(n_samples, in_channels, image_size, image_size)
    probs = torch.ones(num_classes)
    probs[class_bias % num_classes] = 5.0
    probs = probs / probs.sum()
    y = torch.multinomial(probs, num_samples=n_samples, replacement=True)
    return TensorDataset(x, y)




@dataclass
class ServiceState:
    energy_budget: float   # B_k^t
    compute_budget: float  # C_k^t (FLOPs limit)
    memory_budget: float   # M_k^t (params limit)
    alpha: float           # energy coefficient
    local_steps: int       # T_k^t



@torch.no_grad()
def evaluate_model(model: nn.Module, loader: DataLoader) -> float:
    model.eval()
    total = 0
    correct = 0
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        logits = model(x)
        preds = logits.argmax(dim=1)
        total += y.size(0)
        correct += (preds == y).sum().item()
    return 100.0 * correct / max(total, 1)




def split_shared_and_personal_params(model: OrchNASModel):
    shared = list(model.backbone.parameters())
    personal = list(model.personal_head.parameters())
    return shared, personal


def train_global_backbone_one_epoch(
    model: OrchNASModel,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
):
    model.train()
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        logits = model(x)
        loss = F.cross_entropy(logits, y)
        loss.backward()
        optimizer.step()


def train_personalised_with_prox_and_dual(
    global_model: OrchNASModel,
    local_model: OrchNASModel,
    loader: DataLoader,
    mu: float,
    dual_eta: float,
    energy_term: float,
    lr: float = 1e-3,
):

    local_model.train()
    opt = torch.optim.Adam(local_model.personal_head.parameters(), lr=lr)

    global_head = copy.deepcopy(local_model.personal_head.state_dict())  # prox anchor for head-like regularization

    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        opt.zero_grad()
        logits = local_model(x)
        ce = F.cross_entropy(logits, y)

        prox = 0.0
        for name, p in local_model.personal_head.named_parameters():
            prox = prox + torch.sum((p - global_head[name].to(DEVICE)) ** 2)

        loss = ce + (mu / 2.0) * prox + dual_eta * energy_term * 0.0
        # energy term has no gradient wrt personal head; kept conceptually
        loss.backward()
        opt.step()




def prune_width(width: int) -> int:
    choices = SEARCH_SPACE["width"]
    idx = choices.index(width)
    return choices[max(0, idx - 1)]


def prune_depth(depth: int) -> int:
    choices = SEARCH_SPACE["depth"]
    idx = choices.index(depth)
    return choices[max(0, idx - 1)]


def simpler_op(op_type: str) -> str:
    # crude ordering from heavier -> lighter
    order = ["inverted_bottleneck", "standard", "depthwise"]
    idx = order.index(op_type)
    return order[min(idx + 1, len(order) - 1)]


def smaller_kernel(k: int) -> int:
    choices = SEARCH_SPACE["kernel_size"]
    idx = choices.index(k)
    return choices[max(0, idx - 1)]


def generate_pruning_moves(arch: Dict[str, Any]) -> List[Tuple[str, Dict[str, Any]]]:
    moves = []

    if arch["width"] != min(SEARCH_SPACE["width"]):
        a = copy.deepcopy(arch)
        a["width"] = prune_width(a["width"])
        moves.append(("width", a))

    if arch["depth"] != min(SEARCH_SPACE["depth"]):
        a = copy.deepcopy(arch)
        a["depth"] = prune_depth(a["depth"])
        moves.append(("depth", a))

    if arch["op_type"] != "depthwise":
        a = copy.deepcopy(arch)
        a["op_type"] = simpler_op(a["op_type"])
        moves.append(("op_type", a))

    if arch["kernel_size"] != min(SEARCH_SPACE["kernel_size"]):
        a = copy.deepcopy(arch)
        a["kernel_size"] = smaller_kernel(a["kernel_size"])
        moves.append(("kernel_size", a))

    return moves


def is_feasible(arch: Dict[str, Any], state: ServiceState, num_classes: int = 10) -> bool:
    model = OrchNASModel(arch, num_classes=num_classes).to(DEVICE)
    params = count_parameters(model)
    flops = estimate_flops(arch, num_classes=num_classes)
    energy = estimate_energy(arch, state.alpha, state.local_steps)

    return (
        energy <= state.energy_budget and
        flops <= state.compute_budget and
        params <= state.memory_budget
    )


def greedy_energy_aware_pruning(
    arch: Dict[str, Any],
    val_loader: DataLoader,
    state: ServiceState,
    beta: float = 1e-12,
    num_classes: int = 10,
) -> Dict[str, Any]:

    current_arch = copy.deepcopy(arch)

    def score(base_acc, cand_acc, base_energy, cand_energy):
        delta_acc = base_acc - cand_acc
        delta_energy = base_energy - cand_energy
        return delta_acc - beta * delta_energy

    while not is_feasible(current_arch, state, num_classes=num_classes):
        base_model = OrchNASModel(current_arch, num_classes=num_classes).to(DEVICE)
        base_acc = evaluate_model(base_model, val_loader)
        base_energy = estimate_energy(current_arch, state.alpha, state.local_steps)

        moves = generate_pruning_moves(current_arch)
        if not moves:
            break

        best_score = float("inf")
        best_arch = None

        for _, cand_arch in moves:
            cand_model = OrchNASModel(cand_arch, num_classes=num_classes).to(DEVICE)
            cand_acc = evaluate_model(cand_model, val_loader)
            cand_energy = estimate_energy(cand_arch, state.alpha, state.local_steps)
            s = score(base_acc, cand_acc, base_energy, cand_energy)

            if s < best_score:
                best_score = s
                best_arch = cand_arch

        if best_arch is None:
            break
        current_arch = best_arch

    return current_arch




class EdgeService:
    def __init__(
        self,
        service_id: int,
        train_loader: DataLoader,
        val_loader: DataLoader,
        state: ServiceState,
        num_classes: int = 10,
    ):
        self.service_id = service_id
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.state = state
        self.num_classes = num_classes

        self.personal_arch = random_architecture()
        self.personal_model = OrchNASModel(self.personal_arch, num_classes=num_classes).to(DEVICE)
        self.dual_eta = 0.0

    def dynamic_state(self) -> ServiceState:

        s = copy.deepcopy(self.state)
        # small random dynamics
        s.energy_budget *= random.uniform(0.95, 1.05)
        s.compute_budget *= random.uniform(0.95, 1.05)
        s.memory_budget *= random.uniform(0.95, 1.05)
        s.alpha *= random.uniform(0.98, 1.02)
        return s

    def score_candidate_architectures(
        self,
        candidate_pool: List[Dict[str, Any]],
        lambda_energy: float = 1e-12,
    ) -> List[float]:
        """
        S_k^t(a) = ValAcc_k^t(a) - lambda * E_hat_k^t(a)
        """
        scores = []
        state = self.dynamic_state()

        for arch in candidate_pool:
            model = OrchNASModel(arch, num_classes=self.num_classes).to(DEVICE)
            val_acc = evaluate_model(model, self.val_loader)
            energy = estimate_energy(arch, state.alpha, state.local_steps)
            score = val_acc - lambda_energy * energy
            scores.append(score)

        return scores

    def local_update_from_global(
        self,
        global_arch: Dict[str, Any],
        global_backbone_state: Dict[str, torch.Tensor],
        mu: float,
        rho: float,
        beta: float,
        lr_global: float,
        lr_personal: float,
    ) -> Tuple[Dict[str, torch.Tensor], int, Dict[str, Any], float]:
        """
        Returns:
            updated_backbone_state,
            num_samples,
            final_personal_arch,
            local_energy
        """
        state = self.dynamic_state()

        # Start from global architecture
        local_arch = copy.deepcopy(global_arch)

        # If infeasible, prune
        if not is_feasible(local_arch, state, num_classes=self.num_classes):
            local_arch = greedy_energy_aware_pruning(
                local_arch,
                self.val_loader,
                state,
                beta=beta,
                num_classes=self.num_classes,
            )

        # Build local model using pruned architecture
        local_model = OrchNASModel(local_arch, num_classes=self.num_classes).to(DEVICE)

        # Load matching backbone weights where possible
        try:
            local_model.backbone.load_state_dict(global_backbone_state, strict=False)
        except Exception:
            pass

        # 1) Update shared/global backbone
        backbone_opt = torch.optim.SGD(local_model.backbone.parameters(), lr=lr_global, momentum=0.9)
        train_global_backbone_one_epoch(local_model, self.train_loader, backbone_opt)

        # 2) Personalised update with proximal regularisation
        local_energy = estimate_energy(local_arch, state.alpha, state.local_steps)
        train_personalised_with_prox_and_dual(
            global_model=local_model,
            local_model=local_model,
            loader=self.train_loader,
            mu=mu,
            dual_eta=self.dual_eta,
            energy_term=local_energy,
            lr=lr_personal,
        )

        # 3) Dual update: eta <- [eta + rho * (E - B)]_+
        self.dual_eta = max(0.0, self.dual_eta + rho * (local_energy - state.energy_budget))

        # save personal state
        self.personal_arch = local_arch
        self.personal_model = local_model

        num_samples = len(self.train_loader.dataset)
        return copy.deepcopy(local_model.backbone.state_dict()), num_samples, local_arch, local_energy


class NASService:
    def __init__(
        self,
        services: List[EdgeService],
        num_classes: int = 10,
        candidate_pool_size: int = 8,
        lambda_energy: float = 1e-12,
        mu: float = 1e-4,
        rho: float = 1e-12,
        beta: float = 1e-12,
        lr_global: float = 1e-2,
        lr_personal: float = 1e-3,
    ):
        self.services = services
        self.num_classes = num_classes
        self.candidate_pool_size = candidate_pool_size

        self.lambda_energy = lambda_energy
        self.mu = mu
        self.rho = rho
        self.beta = beta
        self.lr_global = lr_global
        self.lr_personal = lr_personal

        self.global_arch = random_architecture()
        self.global_model = OrchNASModel(self.global_arch, num_classes=num_classes).to(DEVICE)

        self.candidate_pool = [random_architecture() for _ in range(candidate_pool_size)]

    def evolve_candidate_pool(self):
        """
        Generate new candidates from current best/global architecture.
        """
        new_pool = [copy.deepcopy(self.global_arch)]
        while len(new_pool) < self.candidate_pool_size:
            new_arch = mutate_architecture(self.global_arch, mutation_prob=0.7)
            if architecture_to_key(new_arch) not in {architecture_to_key(a) for a in new_pool}:
                new_pool.append(new_arch)
        self.candidate_pool = new_pool

    def select_global_architecture(self, selected_services: List[EdgeService]) -> Dict[str, Any]:
        """
        a^t = argmax_a sum_k p_k^t S_k^t(a)
        """
        service_sizes = [len(s.train_loader.dataset) for s in selected_services]
        total = sum(service_sizes)

        weighted_scores = [0.0 for _ in self.candidate_pool]

        for service, size in zip(selected_services, service_sizes):
            pk = size / max(total, 1)
            scores = service.score_candidate_architectures(
                self.candidate_pool,
                lambda_energy=self.lambda_energy,
            )
            for i, s in enumerate(scores):
                weighted_scores[i] += pk * s

        best_idx = int(torch.tensor(weighted_scores).argmax().item())
        return self.candidate_pool[best_idx]

    @staticmethod
    def fedavg_backbone(states: List[Dict[str, torch.Tensor]], weights: List[int]) -> Dict[str, torch.Tensor]:
        total = sum(weights)
        avg_state = copy.deepcopy(states[0])

        for key in avg_state.keys():
            avg_state[key] = sum(state[key] * (w / total) for state, w in zip(states, weights))
        return avg_state

    def train(self, rounds: int = 5, clients_per_round: int = 5):
        for rnd in range(1, rounds + 1):
            print(f"\n========== Round {rnd} ==========")

            selected_services = random.sample(
                self.services,
                k=min(clients_per_round, len(self.services))
            )

            # Step 1: evolve candidate pool
            self.evolve_candidate_pool()

            # Step 2: select global architecture by energy-aware scoring
            self.global_arch = self.select_global_architecture(selected_services)
            print("Selected global architecture:", self.global_arch)

            # Rebuild global model if architecture changed
            old_backbone_state = copy.deepcopy(self.global_model.backbone.state_dict())
            self.global_model = OrchNASModel(self.global_arch, num_classes=self.num_classes).to(DEVICE)
            try:
                self.global_model.backbone.load_state_dict(old_backbone_state, strict=False)
            except Exception:
                pass

            # Step 3: local updates
            local_states = []
            local_weights = []
            local_energies = []

            for service in selected_services:
                state_dict, n, local_arch, local_energy = service.local_update_from_global(
                    global_arch=self.global_arch,
                    global_backbone_state=copy.deepcopy(self.global_model.backbone.state_dict()),
                    mu=self.mu,
                    rho=self.rho,
                    beta=self.beta,
                    lr_global=self.lr_global,
                    lr_personal=self.lr_personal,
                )

                print(
                    f"Service {service.service_id} | "
                    f"arch={local_arch} | energy={local_energy:.4e} | eta={service.dual_eta:.4e}"
                )
                local_states.append(state_dict)
                local_weights.append(n)
                local_energies.append(local_energy)

            # Step 4: global aggregation
            avg_state = self.fedavg_backbone(local_states, local_weights)
            self.global_model.backbone.load_state_dict(avg_state, strict=False)

            avg_energy = sum(local_energies) / max(len(local_energies), 1)
            print(f"Average local energy this round: {avg_energy:.4e}")

    def evaluate_global(self, loader: DataLoader) -> float:
        return evaluate_model(self.global_model, loader)



def build_demo_services(
    num_services: int = 10,
    batch_size: int = 32,
    num_classes: int = 10,
) -> List[EdgeService]:
    services = []

    for k in range(num_services):
        train_ds = make_synthetic_client_dataset(
            n_samples=256,
            num_classes=num_classes,
            class_bias=k % num_classes,
        )
        val_ds = make_synthetic_client_dataset(
            n_samples=128,
            num_classes=num_classes,
            class_bias=(k + 1) % num_classes,
        )

        train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
        val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)

        # low / medium / high resource profiles
        resource_group = k % 3
        if resource_group == 0:
            state = ServiceState(
                energy_budget=2.5e9,
                compute_budget=2.0e9,
                memory_budget=2_000_000,
                alpha=1e-6,
                local_steps=20,
            )
        elif resource_group == 1:
            state = ServiceState(
                energy_budget=5.0e9,
                compute_budget=4.0e9,
                memory_budget=4_000_000,
                alpha=8e-7,
                local_steps=20,
            )
        else:
            state = ServiceState(
                energy_budget=8.0e9,
                compute_budget=6.5e9,
                memory_budget=8_000_000,
                alpha=6e-7,
                local_steps=20,
            )

        services.append(
            EdgeService(
                service_id=k,
                train_loader=train_loader,
                val_loader=val_loader,
                state=state,
                num_classes=num_classes,
            )
        )

    return services


def main():
    services = build_demo_services(num_services=12, batch_size=32, num_classes=10)

    orchestrator = NASService(
        services=services,
        num_classes=10,
        candidate_pool_size=6,
        lambda_energy=1e-10,   # tune this
        mu=1e-4,
        rho=1e-10,
        beta=1e-10,
        lr_global=1e-2,
        lr_personal=1e-3,
    )

    test_ds = make_synthetic_client_dataset(n_samples=256, num_classes=10, class_bias=0)
    test_loader = DataLoader(test_ds, batch_size=64, shuffle=False)

    print("Initial global accuracy:", orchestrator.evaluate_global(test_loader))
    orchestrator.train(rounds=5, clients_per_round=4)
    print("Final global accuracy:", orchestrator.evaluate_global(test_loader))


if __name__ == "__main__":
    main()